# D04 — Reference single-cell datasets

Documents the two immune datasets used by the matched-control deletion test
(Section 2.4 of the manuscript) and shows how to check a local copy.

Acquisition itself is performed by `analysis/E2_downstream_ablation.py --setup`,
which downloads, tokenises and subsamples in one pass and writes its caches to
`cache/`. This notebook is the provenance record and a verification aid; it does
not need to be run to reproduce anything.

| Dataset | Source | Cells used | Cell types |
|---|---|---:|---:|
| PBMC3k | 10x Genomics, via `scanpy.datasets` | 2,638 | 8 |
| Tabula Sapiens — Immune | CellxGene Discover | 3,919 | 22 of 43 |

**Licences and versions:** see `DATA_MANIFEST.md`.


## 1. PBMC3k

The 10x Genomics PBMC3k dataset, distributed with scanpy. 2,638 cells remain
after the standard quality filter, annotated into 8 immune types by the
`louvain` labels shipped with the processed object. It is the primary dataset
for the deletion test.


In [ ]:
from pathlib import Path

import scanpy as sc

CACHE = Path("..") / "cache"

pbmc = sc.datasets.pbmc3k_processed()
print(f"cells      : {pbmc.n_obs:,}")
print(f"genes      : {pbmc.n_vars:,}")
print(f"cell types : {pbmc.obs['louvain'].nunique()}")
print(pbmc.obs["louvain"].value_counts().to_string())


## 2. Tabula Sapiens — Immune

CellxGene Discover dataset `78b60b70-129a-4a6d-b15f-825b241eec66`, in collection
`e5f58829-1a66-40b5-a624-9046778e74f5`. The full immune tissue object holds
19,984 cells.

`E2_downstream_ablation.py --setup --datasets tabula_sapiens --subsample 4000`
fetches it, preferring `cellxgene-census` streaming and falling back to the
CellxGene asset API, then draws a stratified subsample of 4,000 cells and drops
cell types with fewer than 10 cells (`MIN_CELLS_PER_TYPE = 10`). 3,919 cells
across 22 of the 43 annotated types survive that filter, and those are the cells
analysed. The subsample size is recorded in the tokenisation cache and a run
that requests a different size is refused rather than silently reusing it.

This is the replication arm reported in Section 3.4. It is not reachable from
`run_all.sh`; see `MANUSCRIPT_TRACEABILITY.md` for the exact command.


In [ ]:
# Reports on a local copy if --setup has already fetched one. Downloads nothing.
h5ad = CACHE / "E2_tabula_sapiens_immune.h5ad"

if not h5ad.exists():
    print(f"No local copy at {h5ad}.")
    print("Fetch it with:")
    print("  uv run python analysis/E2_downstream_ablation.py --setup \\")
    print("      --datasets tabula_sapiens --subsample 4000")
else:
    import anndata as ad

    ts = ad.read_h5ad(h5ad)
    print(f"cells      : {ts.n_obs:,}")
    print(f"genes      : {ts.n_vars:,}")
    for col in ("cell_type", "cell_ontology_class"):
        if col in ts.obs:
            print(f"cell types : {ts.obs[col].nunique()}  (from obs['{col}'])")
            break


## 3. What the deletion test consumes

`--setup` writes the following to `cache/`, none of which is shipped because of
size:

- `E2_<dataset>_tokenized.json` — rank-value encoded cells
- `E2_<dataset>_gene_stats.csv` — per-gene mean expression and breadth in that dataset
- `E2_matched_controls_<dataset>*.json` — the matched-control draws, **which are shipped**
- `E2_matched_controls_<dataset>*_spec.json` — the specification each draw set was built under

The matched-control draws and their specification sidecars are committed, so the
null can be inspected and re-validated without re-running the 18-hour ablation.
Every draw is checked on load for size, duplicate genes, overlap with the
treatment set and exact mutually exclusive class composition.
